# 1-D reflected-beam electric-field reconstruction

This notebook reproduces the ray-based **field-limiter** side of the one-dimensional reflected-beam test in Sec. III A and Fig. 1 of Follett *et al.*, *Physics of Plasmas* **29**, 113902 (2022), using the published case described in [Follett et al. (2022)](https://doi.org/10.1063/5.0123462).

A normally incident 351 nm ray propagates from $x=-16\,\mu\mathrm{m}$ into the paper's $S=1/16$ LILAC power-law density profile. It reaches $n_e/n_\mathrm{crit}=1$, forms a fold caustic, and returns along a second sheet. Inverse bremsstrahlung is disabled. The notebook interpolates both sheets to the 1-D cell centres and coherently reconstructs the complex electric field.

> The project currently implements the paper's improved field limiter (its Eq. 11), not the etalon-integral or LPSE solutions also shown in Fig. 1. The uncapped curve is included to expose the geometrical-optics divergence.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd()
if not (repo_root / "configs").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "configs").is_dir():
    raise RuntimeError("Run this notebook from the repository root or examples folder")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from examples._workflows import run_one_dimensional_field_reconstruction
from pyGATH.raytracing import RAY_STATE_LAYOUT, critical_density

In [ ]:
config_path = (
    repo_root
    / "configs"
    / "example_configs"
    / "one_dimensional_field_reconstruction.toml"
)

started = time.perf_counter()
result, grid, beams, reconstruction, checks = run_one_dimensional_field_reconstruction(
    config_path
)
elapsed = time.perf_counter() - started
print(f"Trace and reconstruction completed in {elapsed:.2f} s")
print(f"Sheet fields: {result.sheet_fields.shape}")
print(f"Analytical critical surface: {checks['expected_critical_x_m'] * 1e6:.5f} um")
print(f"Numerical caustic:          {checks['numerical_caustic_x_m'] * 1e6:.5f} um")
print(f"ne/ncritical at caustic:   {checks['caustic_density_over_ncritical']:.8f}")
print(
    f"Maximum IB deposition:     {checks['maximum_inverse_brems_deposition_w_m3']:.3e} W/m^3"
)

## Critical reflection and the two sheets

The density is the paper's Eq. (13), mapped by $r=36.42\,\mu\mathrm{m}-x$. The primary ray remains natively three-dimensional, but its inactive positions and momenta stay zero. Sheet 1 runs from injection to the caustic; sheet 2 runs from the same caustic back to the injection boundary.

In [ ]:
sheet_fields = np.asarray(result.sheet_fields[0, :, 0, 0])
sheet_x_um = sheet_fields[..., RAY_STATE_LAYOUT.position][..., 0] * 1e6
sheet_path_um = sheet_fields[..., RAY_STATE_LAYOUT.path_length] * 1e6
sheet_momentum_x = sheet_fields[..., RAY_STATE_LAYOUT.momentum][..., 0]

x_m = reconstruction["x_m"]
cartesian_centres = np.stack((x_m, np.zeros_like(x_m), np.zeros_like(x_m)), axis=-1)
density_ratio = np.asarray(grid.interpolate(cartesian_centres).ne) / float(
    critical_density(beams.omega[0])
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
axes[0].plot(x_m * 1e6, density_ratio, color="black")
axes[0].axhline(1.0, color="tab:red", linestyle="--", label=r"$n_e/n_{crit}=1$")
axes[0].axvline(
    checks["numerical_caustic_x_m"] * 1e6,
    color="tab:purple",
    linestyle=":",
    label="detected caustic",
)
axes[0].set(
    xlabel=r"$x$ [$\mu$m]", ylabel=r"$n_e/n_{crit}$", title="Paper density profile"
)
axes[0].legend()

for sheet, label in enumerate(("incident sheet", "reflected sheet")):
    axes[1].plot(sheet_x_um[sheet], sheet_momentum_x[sheet], label=label)
axes[1].axvline(
    checks["numerical_caustic_x_m"] * 1e6, color="tab:purple", linestyle=":"
)
axes[1].set(
    xlabel=r"$x$ [$\mu$m]", ylabel=r"$p_x$", title="Momentum reverses at the caustic"
)
axes[1].legend();

## Coherent reconstruction

The segment field interpolator puts the unwrapped phase length and field amplitude from both sheets onto common cell centres. Following Eq. (4) of the paper,

$$E(x)=\sum_j |E_j(x)|\exp\left[i\left(\frac{\omega}{c}\ell_{\phi,j}(x)-\frac{\pi\alpha_j}{2}\right)\right],$$

with $\alpha_1=0$ for the incident sheet and $\alpha_2=1$ after the caustic. The absolute phase origin is arbitrary and has been set at the first incident cell. The capped reconstruction uses the improved field limiter already calculated by the tracer; the diagnostic uncapped reconstruction uses the raw geometrical-optics amplitude. Every plotted field is converted from V/m to the dimensionless quiver amplitude $a=eE/(m_e c\omega_0)$.

In [ ]:
x_um = reconstruction["x_m"] * 1e6
to_quiver_amplitude = reconstruction["quiver_amplitude_per_v_m"]
uncapped_sheets = reconstruction["uncapped_sheet_magnitude_v_m"] * to_quiver_amplitude
capped_sheets = reconstruction["capped_sheet_magnitude_v_m"] * to_quiver_amplitude
uncapped_field = reconstruction["uncapped_field_v_m"] * to_quiver_amplitude
capped_field = reconstruction["capped_field_v_m"] * to_quiver_amplitude
caustic_um = checks["numerical_caustic_x_m"] * 1e6
near_caustic = (x_um > caustic_um - 2.0) & (x_um < caustic_um + 0.15)

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for sheet, color in enumerate(("tab:blue", "tab:orange")):
    axes[0, 0].plot(
        x_um,
        uncapped_sheets[sheet],
        color=color,
        alpha=0.35,
        linestyle=":",
        label=f"sheet {sheet + 1} uncapped",
    )
    axes[0, 0].plot(
        x_um, capped_sheets[sheet], color=color, label=f"sheet {sheet + 1} capped"
    )
axes[0, 0].set(
    xlabel=r"$x$ [$\mu$m]",
    ylabel=r"$e|E_j|/(m_e c \omega_0)$",
    title="Individual sheet quiver amplitudes",
)
axes[0, 0].legend(ncol=2, fontsize=8)

axes[0, 1].plot(
    x_um, np.real(uncapped_field), color="0.65", linewidth=0.8, label="uncapped"
)
axes[0, 1].plot(
    x_um, np.real(capped_field), color="tab:blue", linewidth=0.9, label="field limiter"
)
axes[0, 1].set(
    xlabel=r"$x$ [$\mu$m]",
    ylabel=r"$e\,\mathrm{Re}(E)/(m_e c \omega_0)$",
    title="Coherent incident + reflected quiver field",
)
axes[0, 1].legend()

axes[1, 0].plot(
    x_um[near_caustic],
    np.real(uncapped_field[near_caustic]),
    color="0.6",
    linestyle=":",
    label="uncapped",
)
axes[1, 0].plot(
    x_um[near_caustic],
    np.real(capped_field[near_caustic]),
    color="tab:blue",
    label="field limiter",
)
axes[1, 0].axvline(caustic_um, color="tab:purple", linestyle=":", label="caustic")
axes[1, 0].set(
    xlabel=r"$x$ [$\mu$m]",
    ylabel=r"$e\,\mathrm{Re}(E)/(m_e c \omega_0)$",
    title="Caustic-region quiver field",
)
axes[1, 0].legend()

axes[1, 1].plot(
    x_um[near_caustic],
    np.abs(uncapped_field[near_caustic]),
    color="0.6",
    linestyle=":",
    label="uncapped",
)
axes[1, 1].plot(
    x_um[near_caustic],
    np.abs(capped_field[near_caustic]),
    color="tab:red",
    label="field limiter",
)
axes[1, 1].axvline(caustic_um, color="tab:purple", linestyle=":")
axes[1, 1].set(
    xlabel=r"$x$ [$\mu$m]",
    ylabel=r"$e|E|/(m_e c \omega_0)$",
    title="Finite coherent quiver amplitude near reflection",
)
axes[1, 1].legend();

In [ ]:
active = np.any(reconstruction["inside"], axis=0)
assert np.all(np.isfinite(capped_field[active]))
assert np.max(uncapped_sheets[:, active]) > np.max(capped_sheets[:, active])
assert checks["maximum_inverse_brems_deposition_w_m3"] == 0.0
assert np.allclose(sheet_fields[..., RAY_STATE_LAYOUT.position][..., 1:], 0.0)
assert np.allclose(sheet_fields[..., RAY_STATE_LAYOUT.momentum][..., 1:], 0.0)
print("All 1-D reflection and reconstruction checks passed.")